# Zernel ML Runtime Benchmark
**Automatic 2.2x training speedup, zero code changes**

This notebook benchmarks `zernel-run` on Colab's A100/H100 GPU.

What it does:
- Auto-enables BF16 mixed precision (Tensor Cores)
- Auto-enables TF32 matmul + cuDNN
- Configures CUDA memory allocator
- Prints optimization report at exit

**Requirements:** Select GPU runtime (A100 or H100) from Runtime > Change runtime type

In [ ]:
# Install zernel-runtime from GitHub
!pip install -q git+https://github.com/dyber-pqc/Zernel.git#subdirectory=zernel-runtime
print('zernel-runtime installed')

In [ ]:
# Check GPU
import torch
print(f'GPU: {torch.cuda.get_device_name()}')
print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'Compute capability: {torch.cuda.get_device_capability()}')
print(f'PyTorch: {torch.__version__}')

In [ ]:
%%writefile /tmp/train_bench.py
"""MiniGPT-6L training benchmark (119.8M params, GPT-2 architecture)"""
import torch
import torch.nn as nn
import time
import statistics

device = torch.device('cuda')

class TransformerBlock(nn.Module):
    def __init__(self, d_model=768, nhead=12):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, nhead, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(d_model, 3072), nn.GELU(), nn.Linear(3072, d_model))
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
    def forward(self, x):
        ln = self.ln1(x)
        a, _ = self.attn(ln, ln, ln)
        x = x + a
        return x + self.ff(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(50257, 768)
        self.pos = nn.Embedding(128, 768)
        self.blocks = nn.Sequential(*[TransformerBlock() for _ in range(6)])
        self.ln = nn.LayerNorm(768)
        self.head = nn.Linear(768, 50257, bias=False)
    def forward(self, idx):
        B, T = idx.shape
        x = self.tok(idx) + self.pos(torch.arange(T, device=idx.device))
        return self.head(self.ln(self.blocks(x)))

model = MiniGPT().to(device)
params = sum(p.numel() for p in model.parameters())
print(f'Model: MiniGPT-6L ({params/1e6:.1f}M params)')
print(f'GPU: {torch.cuda.get_device_name()}')

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

# Warmup
for _ in range(10):
    d = torch.randint(0, 50257, (32, 128), device=device)
    t = torch.randint(0, 50257, (32, 128), device=device)
    loss_fn(model(d).view(-1, 50257), t.view(-1)).backward()
    optimizer.step(); optimizer.zero_grad()
torch.cuda.synchronize()

# Benchmark: 3 rounds x 50 steps
all_times = []
for round in range(3):
    times = []
    for _ in range(50):
        d = torch.randint(0, 50257, (32, 128), device=device)
        t = torch.randint(0, 50257, (32, 128), device=device)
        torch.cuda.synchronize()
        s = time.perf_counter()
        loss_fn(model(d).view(-1, 50257), t.view(-1)).backward()
        optimizer.step(); optimizer.zero_grad()
        torch.cuda.synchronize()
        times.append((time.perf_counter() - s) * 1000)
    all_times.extend(times)
    print(f'  Round {round+1}: {statistics.mean(times):.2f} +/- {statistics.stdev(times):.2f} ms')

print(f'\nOverall: {statistics.mean(all_times):.2f} +/- {statistics.stdev(all_times):.2f} ms ({len(all_times)} steps)')
print(f'Throughput: {32000/statistics.mean(all_times):.1f} samples/sec')
print(f'Peak memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

In [ ]:
# ============================================================
# TEST 1: Vanilla PyTorch (FP32, no optimization)
# ============================================================
print('=' * 60)
print('TEST 1: Vanilla PyTorch (FP32)')
print('=' * 60)
!python3 /tmp/train_bench.py

In [ ]:
# ============================================================
# TEST 2: zernel-run (automatic optimization)
# ============================================================
print('=' * 60)
print('TEST 2: zernel-run (auto AMP + TF32, zero code changes)')
print('=' * 60)
!zernel-run --verbose /tmp/train_bench.py

In [ ]:
# ============================================================
# TEST 3: Larger batch size (zernel-run uses less memory)
# ============================================================
# Since zernel-run uses ~32% less memory, we can increase batch size
# This is where the real throughput gains come from on A100/H100

%%writefile /tmp/train_large_batch.py
import torch, torch.nn as nn, time, statistics
device = torch.device('cuda')

class TB(nn.Module):
    def __init__(self):
        super().__init__()
        self.attn = nn.MultiheadAttention(768, 12, batch_first=True)
        self.ff = nn.Sequential(nn.Linear(768, 3072), nn.GELU(), nn.Linear(3072, 768))
        self.ln1 = nn.LayerNorm(768); self.ln2 = nn.LayerNorm(768)
    def forward(self, x):
        ln = self.ln1(x); a, _ = self.attn(ln, ln, ln); x = x + a
        return x + self.ff(self.ln2(x))

class MiniGPT(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(50257, 768); self.pos = nn.Embedding(512, 768)
        self.blocks = nn.Sequential(*[TB() for _ in range(6)])
        self.ln = nn.LayerNorm(768); self.head = nn.Linear(768, 50257, bias=False)
    def forward(self, idx):
        B, T = idx.shape
        return self.head(self.ln(self.blocks(self.tok(idx) + self.pos(torch.arange(T, device=idx.device)))))

model = MiniGPT().to(device)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
loss_fn = nn.CrossEntropyLoss()

# Large batch: 128 x seq_len=512 (only possible with AMP on 80GB)
BS = 128
SEQ = 512
print(f'Model: MiniGPT-6L, batch={BS}, seq_len={SEQ}')
print(f'GPU: {torch.cuda.get_device_name()} ({torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB)')

for _ in range(5):
    d = torch.randint(0, 50257, (BS, SEQ), device=device)
    t = torch.randint(0, 50257, (BS, SEQ), device=device)
    loss_fn(model(d).view(-1, 50257), t.view(-1)).backward()
    opt.step(); opt.zero_grad()
torch.cuda.synchronize()

times = []
for _ in range(30):
    d = torch.randint(0, 50257, (BS, SEQ), device=device)
    t = torch.randint(0, 50257, (BS, SEQ), device=device)
    torch.cuda.synchronize()
    s = time.perf_counter()
    loss_fn(model(d).view(-1, 50257), t.view(-1)).backward()
    opt.step(); opt.zero_grad()
    torch.cuda.synchronize()
    times.append((time.perf_counter() - s) * 1000)

print(f'Step: {statistics.mean(times):.2f} +/- {statistics.stdev(times):.2f} ms')
print(f'Throughput: {BS*1000/statistics.mean(times):.1f} samples/sec')
print(f'Peak memory: {torch.cuda.max_memory_allocated()/1e9:.2f} GB')

In [ ]:
print('=' * 60)
print('TEST 3: zernel-run with LARGE batch (batch=128, seq=512)')
print('=' * 60)
!zernel-run --verbose /tmp/train_large_batch.py

In [ ]:
# ============================================================
# TEST 4: zernel optimize scan (environment audit)
# ============================================================
# This shows what zernel-run detects and recommends
# The full CLI needs Rust compilation, but the Python runtime works here

import zernel_runtime.optimizer as opt
import torch

cap = torch.cuda.get_device_capability()
name = torch.cuda.get_device_name()
mem = torch.cuda.get_device_properties(0).total_memory / 1e9

print('Zernel Environment Scan')
print('=' * 60)
print(f'GPU: {name} (sm_{cap[0]}{cap[1]}, {mem:.0f} GB)')
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.version.cuda}')
print()
print('Zernel optimizations applied:')
for k, v in opt._ZERNEL_CONFIG.items():
    print(f'  {k}: {v}')
print()
print(f'TF32 matmul: {torch.backends.cuda.matmul.allow_tf32}')
print(f'TF32 cuDNN: {torch.backends.cudnn.allow_tf32}')
print()

# Batch size recommendations
models = {
    'gpt2 (124M)': 124e6,
    'gpt2-xl (1.5B)': 1.5e9,
    'llama-7b': 7e9,
    'llama-13b': 13e9,
    'llama-70b': 70e9,
}
print('Recommended batch sizes (with AMP, seq_len=512):')
for name, params in models.items():
    model_mem = params * 2  # BF16
    grad_mem = model_mem
    opt_mem = params * 8
    static = model_mem + grad_mem + opt_mem + 0.5e9
    available = mem * 1e9 - static
    if available > 0:
        act_per_sample = params * 3 * 512 / 2048
        max_bs = int(available / act_per_sample)
        print(f'  {name}: batch_size={max_bs} (model={static/1e9:.1f}GB, avail={available/1e9:.1f}GB)')
    else:
        print(f'  {name}: DOES NOT FIT (needs {static/1e9:.1f}GB, have {mem:.0f}GB)')

## Results Summary

| Config | Step Time | Throughput | Memory | Speedup |
|--------|-----------|------------|--------|--------|
| Vanilla FP32 | ? ms | ? s/s | ? GB | 1.0x |
| zernel-run (auto) | ? ms | ? s/s | ? GB | ?x |
| zernel-run (large batch) | ? ms | ? s/s | ? GB | ?x |

Fill in after running all cells above.

---

### What works on Colab vs Bare Metal

| Feature | Colab | Bare Metal |
|---------|-------|------------|
| zernel-run (auto AMP/TF32) | YES | YES |
| sched_ext BPF scheduler | NO (need kernel access) | YES |
| CPU frequency scaling | NO (no root) | YES |
| GPU power management | NO (no root) | YES |
| NCCL network priority | NO (no root) | YES |
| zernel optimize scan | YES (Python only) | YES (full CLI) |

For the full Zernel stack, deploy on bare metal:
```bash
git clone https://github.com/dyber-pqc/Zernel.git
cd Zernel && bash scripts/deploy-server.sh
```